# Small-Signal Stability Analysis: Small-Signal Perturbation Response## ObjectiveAssesses dynamic response to small disturbancesProject: 39 Bus New England System - 2**Study Case: Study Cases 1. Power Flow****Objective:**- Assess dynamic response to small disturbances- Apply a small-signal perturbation (e.g., step change in load or reference power)- Plot generator speed deviations for both scenarios- Outputs: Generator speed vs. time plots, Scenario comparison---

## Step 1: Access PowerFactoryFirst, we need to set up the Python environment to access DIgSILENT PowerFactory.

In [ ]:
# ============================================================================# STEP 1: Access PowerFactory# ============================================================================import osos.environ["PATH"] = r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2" + os.environ["PATH"]import syssys.path.append(r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9")# Import powerfactoryimport powerfactory as pfapp = pf.GetApplication()  # Get the application# ============================================================================# STEP 2: Access and activate project# ============================================================================user = app.GetCurrentUser()project = app.ActivateProject("39 Bus New England System - 2")  # Activate the desired projectprj = app.GetActiveProject()print(f"Project activated: {prj.loc_name}")# ============================================================================# STEP 2.5: Activate study case (if needed)# ============================================================================# Try to activate the study case "Study Cases 1. Power Flow"try:    study_cases = prj.GetContents('*.IntCase')    for sc in study_cases:        if '1. Power Flow' in sc.loc_name or 'Power Flow' in sc.loc_name:            sc.Activate()            print(f"Study case activated: {sc.loc_name}")            breakexcept:    print("Note: Using default/active study case")# ============================================================================# STEP 3: Get all relevant objects (generators, loads)# ============================================================================# Create generator dictionarygenerators = app.GetCalcRelevantObjects('*.ElmSym')gen_dict = {}for gen in generators:    gen_dict[gen.loc_name] = gen# Create load dictionaryloads = app.GetCalcRelevantObjects('*.ElmLod')load_dict = {}for load in loads:    load_dict[load.loc_name] = loadprint(f"Found {len(gen_dict)} generators and {len(load_dict)} loads")# ============================================================================# STEP 4: Set up Small-Signal Perturbation - Base Case# ============================================================================print("\n=== Setting up Small-Signal Perturbation - Base Case ===")app.ResetCalculation()# Select a load for small perturbation (e.g., 1% increase)if len(load_dict) > 0:    # Use first available load    test_load = list(load_dict.values())[0]    initial_load_p = test_load.GetAttribute('plini')  # Initial active power in MW    perturbation_pct = 1.0  # 1% perturbation    perturbation_value = initial_load_p * (perturbation_pct / 100.0)        print(f"Selected load: {test_load.loc_name}")    print(f"Initial load: {initial_load_p:.4f} MW")    print(f"Perturbation: {perturbation_value:.4f} MW ({perturbation_pct}%)")        # Create small load perturbation event    event_folder = app.GetFromStudyCase('IntEvt')    event_name = 'Small Signal Perturbation Base'        try:        event_folder.CreateObject('EvtLod', event_name)        load_event = event_folder.GetContents(event_name)[0] if hasattr(event_folder, 'GetContents') else None                if load_event is None:            # Try alternative method            events = event_folder.GetContents()            load_event = events[0] if events else None                if load_event:            load_event.time = 1.0  # Time of the event (1 second)            load_event.p_target = test_load            load_event.iopt_type = 0  # Type of load change            load_event.dP = perturbation_pct  # Small percentage change                        print("Small-signal perturbation event created")    except Exception as e:        print(f"Note: Event creation may need manual setup: {e}")# ============================================================================# STEP 5: Set up results monitoring - Base Case# ============================================================================# Get results file and add generator speed variableselmres = app.GetFromStudyCase('All calculations.ElmRes')elmres.Clear()# Add speed monitoring for all generatorsfor gen_name, gen in gen_dict.items():    try:        elmres.AddVariable(gen, 'm:speed')  # Generator speed in p.u.    except:        try:            elmres.AddVariable(gen, 'm:phi')  # Alternative: rotor angle        except:            passprint(f"Added speed monitoring for {len(gen_dict)} generators")# ============================================================================# STEP 6: Run Dynamic Simulation - Base Case# ============================================================================print("\n=== Running Dynamic Simulation - Base Case ===")# Set initial conditionsini = app.GetFromStudyCase('ComInc')ini.Execute()print("Initial conditions calculated")# Run dynamic simulationsim = app.GetFromStudyCase('ComSim')sim.tstop = 10.0  # Simulation time: 10 secondssim.Execute()print(f"Dynamic simulation completed (t = 0 to {sim.tstop} s)")# ============================================================================# STEP 7: Export Base Case Results# ============================================================================import osscript_dir = os.path.dirname(os.path.abspath(__file__))comres = app.GetFromStudyCase('ComRes')comres.iopt_csel = 0comres.iopt_locn = 1comres.ciopt_head = 1comres.pResult = elmrescomres.ipt_exp = 6  # 6 is for CSV filecomres.f_name = os.path.join(script_dir, 'small_signal_response_base_case.csv')comres.Execute()print(f"Base case results exported to: small_signal_response_base_case.csv")# ============================================================================# STEP 8: New Generation Case Small-Signal Perturbation# ============================================================================print("\n=== Setting up Small-Signal Perturbation - New Generation Case ===")# NOTE: This section should be modified based on how new generation is addedapp.ResetCalculation()# Re-create perturbation event for new generation caseif len(load_dict) > 0:    test_load = list(load_dict.values())[0]    event_folder = app.GetFromStudyCase('IntEvt')    event_name_new = 'Small Signal Perturbation New Gen'        try:        event_folder.CreateObject('EvtLod', event_name_new)        load_event_new = event_folder.GetContents(event_name_new)[0] if hasattr(event_folder, 'GetContents') else None                if load_event_new is None:            events = event_folder.GetContents()            load_event_new = events[-1] if events else None                if load_event_new:            load_event_new.time = 1.0            load_event_new.p_target = test_load            load_event_new.iopt_type = 0            load_event_new.dP = perturbation_pct                        print("Small-signal perturbation event created for New Generation Case")    except Exception as e:        print(f"Note: Event creation may need manual setup: {e}")# Set up results monitoringelmres.Clear()for gen_name, gen in gen_dict.items():    try:        elmres.AddVariable(gen, 'm:speed')    except:        try:            elmres.AddVariable(gen, 'm:phi')        except:            pass# Set initial conditionsini.Execute()# Run dynamic simulationsim.tstop = 10.0sim.Execute()print(f"Dynamic simulation completed for New Generation Case (t = 0 to {sim.tstop} s)")# ============================================================================# STEP 9: Export New Generation Case Results# ============================================================================comres.f_name = os.path.join(script_dir, 'small_signal_response_new_gen_case.csv')comres.Execute()print(f"New generation case results exported to: small_signal_response_new_gen_case.csv")# ============================================================================# STEP 10: Load CSV Data and Create Visualizations# ============================================================================print("\n=== Creating Visualizations ===")try:    import pandas as pd    import matplotlib.pyplot as plt    import seaborn as sns    import numpy as np    from bokeh.plotting import figure, output_file, save    from bokeh.models import ColumnDataSource, HoverTool    from bokeh.layouts import gridplot        # Set style    sns.set_style("whitegrid")    plt.rcParams['figure.figsize'] = (16, 10)        # Read CSV files    base_df = pd.read_csv(os.path.join(script_dir, 'small_signal_response_base_case.csv'))    new_gen_df = pd.read_csv(os.path.join(script_dir, 'small_signal_response_new_gen_case.csv'))        # Get time column (usually first column)    time_col = base_df.columns[0]        # Find all speed/angle columns    speed_cols_base = [col for col in base_df.columns if 'speed' in col.lower() or 'phi' in col.lower()]    speed_cols_new = [col for col in new_gen_df.columns if 'speed' in col.lower() or 'phi' in col.lower()]        if speed_cols_base and speed_cols_new:        # Create visualizations        num_plots = min(len(speed_cols_base), 5)        fig, axes = plt.subplots(num_plots, 1, figsize=(14, 3 * num_plots))        if num_plots == 1:            axes = [axes]                for idx in range(num_plots):            ax = axes[idx]            col_base = speed_cols_base[idx]            col_new = speed_cols_new[idx] if idx < len(speed_cols_new) else speed_cols_base[idx]                        ax.plot(base_df[time_col], base_df[col_base],                    label='Base Case', color='blue', linewidth=2)            ax.plot(new_gen_df[time_col], new_gen_df[col_new],                    label='New Generation Case', color='red', linewidth=2)            ax.set_xlabel('Time (s)', fontsize=10)            ax.set_ylabel('Speed/Angle (p.u./rad)', fontsize=10)            ax.set_title(f'Generator Response: {col_base}', fontsize=11, fontweight='bold')            ax.legend()            ax.grid(True, alpha=0.3)                plt.tight_layout()        plot_path = os.path.join(script_dir, 'small_signal_response_plots.png')        plt.savefig(plot_path, dpi=300, bbox_inches='tight')        plt.close()        print(f"Static plots saved to: small_signal_response_plots.png")                # Interactive Bokeh plot        try:            output_file(os.path.join(script_dir, 'small_signal_response_interactive.html'))                        plots = []            for idx in range(min(3, len(speed_cols_base))):  # Limit to 3 for readability                col_base = speed_cols_base[idx]                col_new = speed_cols_new[idx] if idx < len(speed_cols_new) else col_base                                p = figure(width=900, height=300,                           title=f"Generator Response: {col_base} (Interactive)",                          x_axis_label="Time (s)", y_axis_label="Speed/Angle (p.u./rad)",                          tools="pan,wheel_zoom,box_zoom,reset,hover,save")                                base_source = ColumnDataSource(data=dict(                    time=base_df[time_col],                    value=base_df[col_base]                ))                                new_gen_source = ColumnDataSource(data=dict(                    time=new_gen_df[time_col],                    value=new_gen_df[col_new]                ))                                p.line('time', 'value', source=base_source, color='blue',                       line_width=2, legend_label='Base Case', alpha=0.8)                p.line('time', 'value', source=new_gen_source, color='red',                       line_width=2, legend_label='New Generation Case', alpha=0.8)                                hover = p.select_one(HoverTool)                hover.tooltips = [("Time", "@time{0.00} s"), ("Value", "@value{0.0000}")]                                p.legend.location = "top_right"                plots.append(p)                        grid = gridplot([plots], toolbar_location='right')            save(grid)            print(f"Interactive plots saved to: small_signal_response_interactive.html")        except Exception as e:            print(f"Note: Bokeh interactive plot creation failed: {e}")    except ImportError as e:    print(f"Note: Visualization libraries not available: {e}")    print("Install required packages: pip install matplotlib seaborn pandas bokeh numpy")except Exception as e:    print(f"Note: Error creating plots: {e}")# ============================================================================# STEP 11: Clean up# ============================================================================app.ResetCalculation()# Delete eventstry:    event_folder = app.GetFromStudyCase('IntEvt')    events = event_folder.GetContents()    for event in events:        if 'Small Signal Perturbation' in event.loc_name:            event.Delete()except:    passprint("\n=== Small-Signal Perturbation Analysis completed successfully ===")print(f"Results saved in: {script_dir}")